In [2]:
import pandas as pd
import numpy as np

data = pd.read_excel("overall-predictions.xlsx")
data

,User Story,Oracle Domain,Oracle Tasks & Sensitive Features,Oracle Unique Sensitive Features,ReFair Domain,ReFair Tasks & Sensitive Features,ReFair Unique Sensitive Features
0,A bioinformatics company is using statistical ...,Biology,"{'classification': [], 'graph mining': []}",{},Biology,{},{}
1,A bioinformatics company is using text generat...,Biology,{'graph mining': []},{},Biology,{},{}
2,A bioinformatics company is using visual quest...,Biology,{'classification': []},{},Biology,{'classification': []},{}
3,A bioinformatics researcher is developing a sy...,Biology,{'graph mining': []},{},Biology,{'graph mining': []},{}
4,A bioinformatics researcher is interested in i...,Biology,{},{},Biology,{},{}
...,...,...,...,...,...,...,...
12396,I am a dermatologist who needs a tool to summa...,Dermatology,{},{},Dermatology,{},{}
12397,I work in sports journalism and need a tool to...,Sport,{},{},Sport,{},{}
12398,I work in the movie industry and I need a mach...,Movies,"{'data summarization': ['age', 'gender']}","{'age', 'gender'}",Movies,"{'data summarization': ['age', 'gender']}","{'age', 'gender'}"
12399,Researchers are using bigram models to analyze...,Biology,{},{},Biology,{},{}


In [3]:
def mean_overlap(set1, set2):
    intersection = len(set1.intersection(set2))
    union = len(set1.union(set2))
    mo_score = intersection / union
    return mo_score

def jaccard_overlap(set1, set2):
    intersection = len(set1.intersection(set2))
    jo_score = intersection / len(set1)
    return jo_score

mo_scores = []
jo_scores = []
mojo_scores = []

for us in data.iterrows():
    if type(eval(us[1]["Oracle Unique Sensitive Features"])) == set:
        mo = mean_overlap(eval(us[1]["Oracle Unique Sensitive Features"]),eval(us[1]["ReFair Unique Sensitive Features"]))
        jo = jaccard_overlap(eval(us[1]["Oracle Unique Sensitive Features"]),eval(us[1]["ReFair Unique Sensitive Features"]))
        mo_scores.append(mo)
        jo_scores.append(jo)
        mojo_scores.append(1 - ((jo + mo) / 2))

print(np.mean(mo_scores))
print(np.mean(jo_scores))
np.mean(mojo_scores)

0.9610753972632007
0.9631388652474817


0.037892868744658796

In [4]:
def levenshtein(seq1, seq2):
    
    size_x = len(seq1) + 1
    size_y = len(seq2) + 1
    matrix = np.zeros ((size_x, size_y))
    for x in range(size_x):
        matrix [x, 0] = x
    for y in range(size_y):
        matrix [0, y] = y

    for x in range(1, size_x):
        for y in range(1, size_y):
            if seq1[x-1] == seq2[y-1]:
                matrix [x,y] = min(
                    matrix[x-1, y] + 1,
                    matrix[x-1, y-1],
                    matrix[x, y-1] + 1
                )
            else:
                matrix [x,y] = min(
                    matrix[x-1,y] + 1,
                    matrix[x-1,y-1] + 1,
                    matrix[x,y-1] + 1
                )
                
    return (matrix[size_x - 1, size_y - 1])

In [12]:
import numpy as np
lev_distance_norm = []
lev_dist = []
lev_dist_no_empty = []
empty_oracle = 0
empty_refair = 0
both_empty = 0
for us in data.iterrows():
    oracle = np.sort(np.array(list(eval(us[1]["Oracle Unique Sensitive Features"]))))
    refair = np.sort(np.array(list(eval(us[1]["ReFair Unique Sensitive Features"]))))
    lev_dist.append(levenshtein(oracle,refair))
    if len(oracle)==0:
        empty_oracle += 1
    if len(refair)==0:
        empty_refair += 1
    if len(oracle)==len(refair)==0:
        both_empty +=1
    if len(oracle)>0:
        lev_dist_no_empty.append(levenshtein(oracle,refair))
        # if levenshtein(oracle,refair) == 0:
        #     print(oracle)
        #     print(refair)
        #     print("-------")
    

count_0 = sum(1 for value in lev_dist if value == 0)
count_1 = sum(1 for value in lev_dist if value == 1)
count_2 = sum(1 for value in lev_dist if value == 2)
count_left = sum(1 for value in lev_dist if value > 2)
total = count_0+count_1+count_2+count_left
print("Total instances")
print(total)
print("Empty set of feature in the oracle:")
print(empty_oracle)
print(empty_oracle/total)
print("Empty set of feature in the Refair output:")
print(empty_refair)
print(empty_refair/total)
print("(Oracle, Refair output) pairs providing the same set of features:")
print(count_0)
print(count_0/total)
print("(Oracle, Refair output) pairs differing in the set of features proposed in 1 feature:")
print(count_1)
print(count_1/total)
print("(Oracle, Refair output) pairs differing in the set of features proposed in 2 features:")
print(count_2)
print(count_2/total)
print("(Oracle, Refair output) pairs differing in the set of features proposed in more than 2 features:")
print(count_left)
print(count_left/total)
print("Empty set for both:")
print(both_empty)
print(both_empty/count_0)

print("Instances in which the oracle set of feature is not empty:")
print((total - empty_oracle))

print("(Oracle, Refair output) pairs providing the same set of features with at least 1 feature:")
print(sum(1 for value in lev_dist_no_empty if value == 0))
print(sum(1 for value in lev_dist_no_empty if value == 0)/(total - empty_oracle))

print("(Oracle, Refair output) pairs providing the same set of features with at least 1 feature differing in 1 feature:")
print(sum(1 for value in lev_dist_no_empty if value == 1))
print(sum(1 for value in lev_dist_no_empty if value == 1)/(total - empty_oracle))

print("(Oracle, Refair output) pairs providing the same set of features with at least 1 feature differing in 2 features:")
print(sum(1 for value in lev_dist_no_empty if value == 2))
print(sum(1 for value in lev_dist_no_empty if value == 2)/(total - empty_oracle))

print("(Oracle, Refair output) pairs providing the same set of features with at least 1 feature differing in more than 2 features:")
print(sum(1 for value in lev_dist_no_empty if value > 2))
print(sum(1 for value in lev_dist_no_empty if value > 2)/(total - empty_oracle))


Total instances
12401
Empty set of feature in the oracle:
7457
0.6013224739940327
Empty set of feature in the Refair output:
7416
0.5980162890089509
(Oracle, Refair output) pairs providing the same set of features:
11969
0.9651640996693815
(Oracle, Refair output) pairs differing in the set of features proposed in 1 feature:
41
0.0033061849850818483
(Oracle, Refair output) pairs differing in the set of features proposed in 2 features:
70
0.00564470607209096
(Oracle, Refair output) pairs differing in the set of features proposed in more than 2 features:
321
0.02588500927344569
Empty set for both:
7253
0.6059821204779012
Instances in which the oracle set of feature is not empty:
4944
(Oracle, Refair output) pairs providing the same set of features with at least 1 feature:
4716
0.9538834951456311
(Oracle, Refair output) pairs providing the same set of features with at least 1 feature differing in 1 feature:
26
0.0052588996763754045
(Oracle, Refair output) pairs providing the same set of fe